In [ ]:
import sys

sys.path.append("..")

import yaml
import numpy as np
from PIL import Image

from ultralytics.utils.plotting import Annotator, Colors
from utils.dataloaders import create_dataloader
from utils.general import LOGGER,check_dataset, check_yaml, colorstr, xywh2xyxy
from utils.augmentations import PreAlbumentations


task = "train"
dataset_yaml = "/mnt/Public/yolo/TRAIN_RGB_WK_2025_06_IMAGE_BB_CV/dataset.yaml"
data = check_dataset(dataset_yaml)  # check
hyp_yaml = check_yaml("../data/hyps/hyp.sea-ai.yaml")

with open(hyp_yaml, "r") as f:
    hyp = yaml.safe_load(f)
    hyp["perspective"] = 0.0

LOGGER.info(colorstr("hyperparameters: ") + ", ".join(f"{k}={v}" for k, v in hyp.items()))

dataloader, dataset = create_dataloader(
    data[task],  #
    imgsz=1280,
    batch_size=1,
    stride=1,
    hyp=hyp,
    pad=0.5 if task == "val" else 0.0,
    augment=True if task == "train" else False,
    workers=1,
    prefix=colorstr(f"{task}: "),
)

def draw_labels(img, labels):
    img = img.numpy().transpose(1, 2, 0).astype(np.uint8)
    annotator = Annotator(np.ascontiguousarray(img))
    colors = Colors()
    for cls, bbox in zip(labels[..., 1], xywh2xyxy(labels[..., 2:])):
        bbox *= np.array([img.shape[1], img.shape[0], img.shape[1], img.shape[0]])
        annotator.box_label(bbox, f"{int(cls)}", color=colors(cls))
    return annotator.result()

In [ ]:
dataset.hyp["pre_crop"] = 0.8
dataset.pre_albumentations = PreAlbumentations(size=dataset.img_size, hyp=dataset.hyp)

for img, labels, im_file, _ in dataset:
    print(img.shape, labels.shape)
    break

Image.fromarray(draw_labels(img, labels))